# As 10 perguntas de negócio — SQL de referência (escrito à mão)

Este notebook responde às 10 perguntas do Checkpoint 1 (`docs/perguntas_negocio.md`) com **SQL escrito à mão**,
seguindo as regras de negócio da camada semântica (`src/semantic_layer.yaml`, regras **R01–R14**).

**Para que serve:**
1. Validar que a base responde às perguntas (antes de existir qualquer agente).
2. Testar as **regras de negócio** na prática.
3. Servir de **gabarito**: estes SQLs são a base do `eval/golden_set.csv` (Etapa 4).

**Regras aplicadas em todas as consultas** (ids do `semantic_layer.yaml`)
| Regra | Definição |
|---|---|
| R01 Datas relativas | calculadas pela **última data da base** (30/06/2026), nunca pelo relógio |
| R02 Preço atual (posto/CEP) | última coleta **daquele posto para aquele produto** |
| R03 CEP | comparado só pelos dígitos (com ou sem hífen) |
| R04 Último mês | último mês **com pesquisa** no recorte filtrado |
| R05 Rankings entre locais | último **trimestre** com pesquisa na base, mínimo de **5 postos** |
| R06 Coleta recente | últimos **3 meses** até o último mês com dados na base |
| R07 Coleta antiga | data anterior à janela de coleta recente → **avisar** |
| R08 Etanol compensa | etanol ÷ gasolina ≤ 0,70 (mesmo posto e data) |
| R09 Transparência | **toda resposta mostra a data/período da pesquisa** |
| R13 Preço médio | **média por posto**: 1º a média de cada posto, depois a média dos postos |
| R14 Preços fora do padrão | **não são removidos nem marcados**: o usuário avalia |

Apelidos padrão das tabelas: `f` fato_precos · `t` dim_tempo · `p` dim_posto · `l` dim_local · `pr` dim_produto · `b` dim_bandeira.

## Conexão com o banco

In [1]:
import os
from pathlib import Path
import duckdb
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

# Procura a raiz do projeto (pasta que contém data/processed) a partir da pasta do notebook
raiz = Path.cwd()
while not (raiz / "data" / "processed").exists() and raiz != raiz.parent:
    raiz = raiz.parent

DB_PATH = os.getenv("DB_PATH") or str(raiz / "data" / "processed" / "combustiveis.duckdb")
con = duckdb.connect(DB_PATH, read_only=True)   # somente leitura: o mesmo guardrail que os agentes terão

def q(sql: str) -> pd.DataFrame:
    """Executa um SQL e devolve um DataFrame."""
    return con.sql(sql).df()

print("Banco:", os.path.relpath(DB_PATH, raiz))   # caminho relativo: não expõe a pasta do usuário
q("SHOW TABLES")

Banco: data\processed\combustiveis.duckdb


,name
0,dim_bandeira
1,dim_local
2,dim_posto
3,dim_produto
4,dim_tempo
5,fato_precos


## 1. Qual é o preço atual da gasolina no CEP 03210-001?

**Regras:** R02 (preço atual = **última coleta** daquele posto para a gasolina), R03 (CEP só pelos dígitos)
e R07 (coluna `coleta_antiga`: avisar se a data for anterior à janela de coleta recente).
**Atenção:** aqui a coleta é de 2022 → a resposta deve avisar que o dado é antigo.

In [2]:
q("""
WITH janela AS (                 -- início da janela de "coleta recente" (R06/R07)
    SELECT CAST(date_trunc('month', MAX(data)) - INTERVAL 2 MONTH AS DATE) AS inicio FROM fato_precos
)
SELECT p.razao_social, p.bairro, p.cep, b.bandeira,
       f.valor_venda AS preco_gasolina,
       f.data        AS data_pesquisa,
       f.data < (SELECT inicio FROM janela) AS coleta_antiga          -- R07: avisar o usuário
FROM fato_precos f
JOIN dim_posto p    ON p.cnpj        = f.cnpj
JOIN dim_produto pr ON pr.id_produto = f.id_produto
JOIN dim_bandeira b ON b.id_bandeira = f.id_bandeira               -- bandeira NA DATA da coleta
WHERE regexp_replace(p.cep, '[^0-9]', '', 'g') = regexp_replace('03210-001', '[^0-9]', '', 'g')   -- R03
  AND pr.produto = 'GASOLINA'
QUALIFY ROW_NUMBER() OVER (PARTITION BY f.cnpj ORDER BY f.data DESC) = 1   -- R02: última coleta de cada posto
""")

,razao_social,bairro,cep,bandeira,preco_gasolina,data_pesquisa,coleta_antiga
0,TRAPEZIO AUTO POSTO LTDA,VILA ALPINA,03210-001,RAIZEN,4.940,2022-09-12,True


## 2. Pela regra dos 70%, liste até 10 postos do município de São Paulo onde o etanol compensa na coleta mais recente.

**Regras:** R06 (coleta recente = últimos **3 meses** até o último mês com dados na base, ex.: abr–jun/2026) e
R08 (etanol compensa se `etanol / gasolina ≤ 0,70`, **mesmo posto e mesma data**). Dentro da janela, pega em cada
posto a **data mais recente em que etanol E gasolina foram coletados juntos**.
**Detalhes:** R12 (município **e** UF). R14: preços fora do padrão não são removidos nem marcados.
O desempate final por `razao_social` deixa o resultado do `LIMIT 10` sempre igual (importante para o gabarito).

In [3]:
q("""
WITH janela AS (                -- R06: últimos 3 meses até o último mês com dados
    SELECT CAST(date_trunc('month', MAX(data)) - INTERVAL 2 MONTH AS DATE) AS inicio,
           MAX(data)                                                      AS fim
    FROM fato_precos
),
precos_dia AS (                 -- R08: etanol e gasolina do MESMO posto na MESMA data
    SELECT f.cnpj, f.data,
           MAX(f.valor_venda) FILTER (WHERE pr.produto = 'ETANOL')   AS etanol,
           MAX(f.valor_venda) FILTER (WHERE pr.produto = 'GASOLINA') AS gasolina
    FROM fato_precos f
    JOIN dim_produto pr ON pr.id_produto = f.id_produto
    JOIN dim_local l    ON l.id_local    = f.id_local
    JOIN janela j       ON f.data BETWEEN j.inicio AND j.fim
    WHERE l.uf = 'SP' AND l.municipio = 'SAO PAULO'
      AND pr.produto IN ('ETANOL', 'GASOLINA')
    GROUP BY f.cnpj, f.data
    HAVING etanol IS NOT NULL AND gasolina IS NOT NULL
),
mais_recente AS (               -- data mais recente de cada posto com os dois produtos
    SELECT *, ROW_NUMBER() OVER (PARTITION BY cnpj ORDER BY data DESC) AS ordem
    FROM precos_dia
)
SELECT p.razao_social, p.bairro,
       m.etanol, m.gasolina,
       ROUND(m.etanol / m.gasolina, 3) AS razao_etanol_gasolina,
       m.data AS data_pesquisa
FROM mais_recente m
JOIN dim_posto p ON p.cnpj = m.cnpj
WHERE m.ordem = 1
  AND m.etanol / m.gasolina <= 0.70
ORDER BY razao_etanol_gasolina, m.data DESC, p.razao_social
LIMIT 10
""")

,razao_social,bairro,etanol,gasolina,razao_etanol_gasolina,data_pesquisa
0,CENTRO AUTOMOTIVO LYON LTDA,PQ BRISTOL,3.970,7.690,0.516,2026-06-29
1,AUTO POSTO TEREZA LTDA.,CAPAO REDONDO,3.990,7.490,0.533,2026-06-30
2,BG SUL PETROLEO LTDA,FERREIRA,3.590,6.590,0.545,2026-06-30
3,CENTRO AUTOMOTIVO R 2 REQUINTE LTDA,JARDIM ANGELA,3.590,6.590,0.545,2026-06-22
4,AUTO POSTO LIDER-BANDEIRAS LTDA,BOM RETIRO,3.290,5.990,0.549,2026-06-24
5,AUTO POSTO MARIA VITORIA LIMITADA,JARDIM ELEDY,3.470,6.270,0.553,2026-06-30
6,CENTRO AUTOMOTIVO LAGOS ANDINOS LTDA,SAUDE,3.470,6.270,0.553,2026-06-29
7,SENA PARK AUTO POSTO LTDA,VL CLEMENTINO,4.970,8.970,0.554,2026-06-29
8,AUTO POSTO CATANIA LTDA,CIDADE ADEMAR,3.380,6.090,0.555,2026-06-22
9,AUTO POSTO CITY TATUAPE LTDA,TATUAPE,3.790,6.790,0.558,2026-06-16


## 3. Qual município do Brasil tem a maior média de preço atual da gasolina?

**Regras:** R05 (todos os municípios no **mesmo período** — o último trimestre com pesquisa na base —
e só municípios com **≥ 5 postos**) e R13 (**média por posto**: 1º a média de cada posto no trimestre,
depois a média dos postos do município). Mostra o período usado (R09).

In [4]:
q("""
WITH periodo AS (               -- R05: último trimestre com pesquisa na base
    SELECT year(MAX(data)) AS ano, quarter(MAX(data)) AS tri FROM fato_precos
),
por_posto AS (                  -- R13, 1º passo: média de cada posto no trimestre
    SELECT l.uf, l.municipio, f.cnpj, AVG(f.valor_venda) AS preco_posto,
           MIN(f.data) AS d1, MAX(f.data) AS d2
    FROM fato_precos f
    JOIN dim_produto pr ON pr.id_produto = f.id_produto
    JOIN dim_local l    ON l.id_local    = f.id_local
    JOIN dim_tempo t    ON t.data        = f.data
    JOIN periodo        ON t.ano = periodo.ano AND t.trimestre = periodo.tri
    WHERE pr.produto = 'GASOLINA'
    GROUP BY l.uf, l.municipio, f.cnpj
)
SELECT uf, municipio,           -- R13, 2º passo: média dos postos; R05: só com >= 5 postos
       ROUND(AVG(preco_posto), 3) AS preco_medio_gasolina,
       COUNT(*)                   AS postos_pesquisados,
       MIN(d1) AS primeira_coleta, MAX(d2) AS ultima_coleta
FROM por_posto
GROUP BY uf, municipio
HAVING COUNT(*) >= 5
ORDER BY preco_medio_gasolina DESC
LIMIT 1
""")

,uf,municipio,preco_medio_gasolina,postos_pesquisados,primeira_coleta,ultima_coleta
0,AM,PARINTINS,9.062,9,2026-04-01,2026-06-22


## 4. Qual município do Brasil tem a menor média de preço atual do etanol?

Mesma lógica da pergunta 3 (R05: último trimestre, ≥ 5 postos; R13: média por posto), produto **ETANOL**, ordem **crescente**.

In [5]:
q("""
WITH periodo AS (               -- R05: último trimestre com pesquisa na base
    SELECT year(MAX(data)) AS ano, quarter(MAX(data)) AS tri FROM fato_precos
),
por_posto AS (                  -- R13, 1º passo: média de cada posto no trimestre
    SELECT l.uf, l.municipio, f.cnpj, AVG(f.valor_venda) AS preco_posto,
           MIN(f.data) AS d1, MAX(f.data) AS d2
    FROM fato_precos f
    JOIN dim_produto pr ON pr.id_produto = f.id_produto
    JOIN dim_local l    ON l.id_local    = f.id_local
    JOIN dim_tempo t    ON t.data        = f.data
    JOIN periodo        ON t.ano = periodo.ano AND t.trimestre = periodo.tri
    WHERE pr.produto = 'ETANOL'
    GROUP BY l.uf, l.municipio, f.cnpj
)
SELECT uf, municipio,           -- R13, 2º passo: média dos postos; R05: só com >= 5 postos
       ROUND(AVG(preco_posto), 3) AS preco_medio_etanol,
       COUNT(*)                   AS postos_pesquisados,
       MIN(d1) AS primeira_coleta, MAX(d2) AS ultima_coleta
FROM por_posto
GROUP BY uf, municipio
HAVING COUNT(*) >= 5
ORDER BY preco_medio_etanol ASC
LIMIT 1
""")

,uf,municipio,preco_medio_etanol,postos_pesquisados,primeira_coleta,ultima_coleta
0,SP,LINS,3.458,15,2026-04-07,2026-06-29


## 5. Qual é o preço médio do diesel S10 no estado do Paraná no último mês com pesquisa?

**Regras:** R04 ("último mês" = último mês **com pesquisa** de diesel S10 **no Paraná**) e R13 (média por posto).

In [6]:
q("""
WITH recorte AS (
    SELECT f.cnpj, f.data, f.valor_venda, t.ano_mes
    FROM fato_precos f
    JOIN dim_produto pr ON pr.id_produto = f.id_produto
    JOIN dim_local l    ON l.id_local    = f.id_local
    JOIN dim_tempo t    ON t.data        = f.data
    WHERE l.uf = 'PR' AND pr.produto = 'DIESEL S10'
),
ultimo_mes AS (                 -- R04: último mês com pesquisa NO RECORTE
    SELECT * FROM recorte WHERE ano_mes = (SELECT MAX(ano_mes) FROM recorte)
),
por_posto AS (                  -- R13: média de cada posto no mês
    SELECT cnpj, AVG(valor_venda) AS preco_posto FROM ultimo_mes GROUP BY cnpj
)
SELECT MAX(ano_mes)                                       AS mes_pesquisa,
       (SELECT ROUND(AVG(preco_posto), 3) FROM por_posto) AS preco_medio_diesel_s10,
       COUNT(DISTINCT cnpj)                               AS postos_pesquisados,
       MIN(data) AS primeira_coleta, MAX(data) AS ultima_coleta
FROM ultimo_mes
""")

,mes_pesquisa,preco_medio_diesel_s10,postos_pesquisados,primeira_coleta,ultima_coleta
0,2026-06,7.044,263,2026-06-01,2026-06-30


## 6. Quantos postos foram pesquisados em Curitiba no último mês com pesquisa?

**Regra:** conta **postos distintos (CNPJ)**, qualquer produto. Último mês com pesquisa **em Curitiba**.

In [7]:
q("""
WITH recorte AS (
    SELECT f.cnpj, f.data, t.ano_mes
    FROM fato_precos f
    JOIN dim_tempo t ON t.data     = f.data
    JOIN dim_local l ON l.id_local = f.id_local
    WHERE l.uf = 'PR' AND l.municipio = 'CURITIBA'          -- R12: município sempre com UF
)
SELECT ano_mes              AS mes_pesquisa,
       COUNT(DISTINCT cnpj) AS postos_pesquisados,        -- postos distintos, não coletas
       MIN(data)            AS primeira_coleta,
       MAX(data)            AS ultima_coleta
FROM recorte
WHERE ano_mes = (SELECT MAX(ano_mes) FROM recorte)        -- R04: último mês com pesquisa em Curitiba
GROUP BY ano_mes
""")

,mes_pesquisa,postos_pesquisados,primeira_coleta,ultima_coleta
0,2026-06,62,2026-06-01,2026-06-24


## 7. Como evoluiu, mês a mês, o preço médio da gasolina no Brasil em 2025?

Série temporal: **média por posto** (R13) em cada `ano_mes`. A variação % em relação ao mês anterior usa a função de janela `LAG`.

In [8]:
q("""
WITH por_posto AS (             -- R13, 1º passo: média de cada posto em cada mês
    SELECT t.ano_mes, f.cnpj, AVG(f.valor_venda) AS preco_posto
    FROM fato_precos f
    JOIN dim_produto pr ON pr.id_produto = f.id_produto
    JOIN dim_tempo t    ON t.data        = f.data
    WHERE pr.produto = 'GASOLINA' AND t.ano = 2025
    GROUP BY t.ano_mes, f.cnpj
),
mensal AS (                     -- R13, 2º passo: média dos postos no mês
    SELECT ano_mes, AVG(preco_posto) AS preco_medio, COUNT(*) AS postos_pesquisados
    FROM por_posto
    GROUP BY ano_mes
)
SELECT ano_mes,
       ROUND(preco_medio, 3) AS preco_medio_gasolina,
       ROUND((preco_medio / LAG(preco_medio) OVER (ORDER BY ano_mes) - 1) * 100, 2) AS variacao_pct_mes_anterior,
       postos_pesquisados
FROM mensal
ORDER BY ano_mes
""")

,ano_mes,preco_medio_gasolina,variacao_pct_mes_anterior,postos_pesquisados
0,2025-01,6.178,NaN,5707
1,2025-02,6.354,2.850,5772
2,2025-03,6.340,-0.230,6024
3,2025-04,6.312,-0.430,5953
4,2025-05,6.280,-0.510,5823
5,2025-06,6.229,-0.800,5578
6,2025-07,6.209,-0.330,5629
7,2025-08,6.177,-0.510,4712
8,2025-09,6.187,0.150,5100
9,2025-10,6.211,0.400,5457


## 8. Postos bandeira branca são mais baratos que os de bandeira? Diferença média da gasolina por região no último mês com pesquisa.

**Regras:** compara a **média por posto** (R13) da gasolina em postos **bandeira branca** × **com bandeira**, por região,
no último mês com pesquisa de gasolina na base. Diferença positiva = bandeira branca mais barata.
A média de cada posto é separada também por bandeira, porque um posto que trocou de bandeira entra em cada grupo
com os preços da sua época. Mostra quantos postos há em cada grupo e o período (R09).

In [9]:
q("""
WITH ultimo AS (                -- último mês com pesquisa de gasolina na base
    SELECT MAX(t.ano_mes) AS ano_mes
    FROM fato_precos f
    JOIN dim_produto pr ON pr.id_produto = f.id_produto
    JOIN dim_tempo t    ON t.data        = f.data
    WHERE pr.produto = 'GASOLINA'
),
por_posto AS (                  -- R13: média de cada posto, separada por bandeira branca / com bandeira
    SELECT l.regiao_nome, f.cnpj, b.eh_bandeira_branca, AVG(f.valor_venda) AS preco_posto,
           MIN(f.data) AS d1, MAX(f.data) AS d2
    FROM fato_precos f
    JOIN dim_produto pr  ON pr.id_produto = f.id_produto
    JOIN dim_local l     ON l.id_local    = f.id_local
    JOIN dim_bandeira b  ON b.id_bandeira = f.id_bandeira
    JOIN dim_tempo t     ON t.data        = f.data
    WHERE pr.produto = 'GASOLINA' AND t.ano_mes = (SELECT ano_mes FROM ultimo)
    GROUP BY l.regiao_nome, f.cnpj, b.eh_bandeira_branca
)
SELECT (SELECT ano_mes FROM ultimo) AS mes_pesquisa, regiao_nome,
       ROUND(AVG(preco_posto) FILTER (WHERE eh_bandeira_branca), 3)     AS media_bandeira_branca,
       ROUND(AVG(preco_posto) FILTER (WHERE NOT eh_bandeira_branca), 3) AS media_com_bandeira,
       ROUND(AVG(preco_posto) FILTER (WHERE NOT eh_bandeira_branca)
           - AVG(preco_posto) FILTER (WHERE eh_bandeira_branca), 3)     AS diferenca_rs,
       ROUND((AVG(preco_posto) FILTER (WHERE NOT eh_bandeira_branca)
            / AVG(preco_posto) FILTER (WHERE eh_bandeira_branca) - 1) * 100, 2) AS diferenca_pct,
       COUNT(*) FILTER (WHERE eh_bandeira_branca)     AS postos_branca,
       COUNT(*) FILTER (WHERE NOT eh_bandeira_branca) AS postos_com_bandeira,
       MIN(d1) AS primeira_coleta, MAX(d2) AS ultima_coleta
FROM por_posto
GROUP BY regiao_nome
ORDER BY diferenca_rs DESC
""")

,mes_pesquisa,regiao_nome,media_bandeira_branca,media_com_bandeira,diferenca_rs,diferenca_pct,postos_branca,postos_com_bandeira,primeira_coleta,ultima_coleta
0,2026-06,Sudeste,6.393,6.542,0.149,2.330,1033,1725,2026-06-01,2026-06-30
1,2026-06,Sul,6.491,6.583,0.093,1.430,249,758,2026-06-01,2026-06-30
2,2026-06,Norte,6.990,7.069,0.079,1.130,120,272,2026-06-01,2026-06-30
3,2026-06,Centro-Oeste,6.667,6.715,0.048,0.730,158,291,2026-06-01,2026-06-30
4,2026-06,Nordeste,7.027,6.917,-0.110,-1.570,449,680,2026-06-01,2026-06-30


## 9. Qual foi a variação % do preço médio da gasolina entre o 1º semestre de 2025 e o 1º semestre de 2026, por estado?

Duas médias por estado (uma por semestre), ambas **por posto** (R13), e o cálculo `(novo / antigo − 1) × 100`.
`FILTER` permite calcular as duas médias numa única passada pelos dados.
Ressalva: os postos pesquisados podem não ser os mesmos nos dois semestres.

In [10]:
q("""
WITH por_posto AS (             -- R13, 1º passo: média de cada posto em cada semestre
    SELECT l.uf, t.ano, f.cnpj, AVG(f.valor_venda) AS preco_posto
    FROM fato_precos f
    JOIN dim_produto pr ON pr.id_produto = f.id_produto
    JOIN dim_local l    ON l.id_local    = f.id_local
    JOIN dim_tempo t    ON t.data        = f.data
    WHERE pr.produto = 'GASOLINA' AND t.semestre = 1 AND t.ano IN (2025, 2026)
    GROUP BY l.uf, t.ano, f.cnpj
)
SELECT uf,                      -- R13, 2º passo: média dos postos em cada semestre
       ROUND(AVG(preco_posto) FILTER (WHERE ano = 2025), 3) AS media_1s_2025,
       ROUND(AVG(preco_posto) FILTER (WHERE ano = 2026), 3) AS media_1s_2026,
       ROUND((AVG(preco_posto) FILTER (WHERE ano = 2026)
            / AVG(preco_posto) FILTER (WHERE ano = 2025) - 1) * 100, 2) AS variacao_pct
FROM por_posto
GROUP BY uf
ORDER BY variacao_pct DESC
""")

,uf,media_1s_2025,media_1s_2026,variacao_pct
0,BA,6.433,7.012,9.000
1,PE,6.343,6.861,8.170
2,AP,6.050,6.510,7.590
3,PI,5.972,6.415,7.410
4,RR,6.986,7.478,7.050
5,SE,6.544,6.965,6.420
6,MT,6.240,6.640,6.410
7,MA,6.067,6.436,6.070
8,RJ,6.164,6.535,6.020
9,TO,6.535,6.896,5.530


## 10. Em Belo Horizonte, qual a diferença entre o posto mais caro e o mais barato do etanol no último mês com pesquisa?

**Amplitude** = máximo − mínimo, sobre as **coletas** (preços reais vistos na bomba, não médias). `arg_max` / `arg_min`
mostram **qual** posto tem cada preço. R04: último mês com pesquisa de etanol **em Belo Horizonte**.

In [11]:
q("""
WITH recorte AS (
    SELECT f.cnpj, f.data, f.valor_venda, t.ano_mes, p.razao_social
    FROM fato_precos f
    JOIN dim_tempo t    ON t.data        = f.data
    JOIN dim_local l    ON l.id_local    = f.id_local
    JOIN dim_produto pr ON pr.id_produto = f.id_produto
    JOIN dim_posto p    ON p.cnpj        = f.cnpj
    WHERE l.uf = 'MG' AND l.municipio = 'BELO HORIZONTE' AND pr.produto = 'ETANOL'
)
SELECT ano_mes AS mes_pesquisa,
       MAX(valor_venda)                     AS preco_mais_caro,
       arg_max(razao_social, valor_venda)   AS posto_mais_caro,
       MIN(valor_venda)                     AS preco_mais_barato,
       arg_min(razao_social, valor_venda)   AS posto_mais_barato,
       MAX(valor_venda) - MIN(valor_venda)  AS amplitude_rs,
       COUNT(DISTINCT cnpj)                 AS postos_pesquisados,
       MIN(data) AS primeira_coleta, MAX(data) AS ultima_coleta
FROM recorte
WHERE ano_mes = (SELECT MAX(ano_mes) FROM recorte)
GROUP BY ano_mes
""")

,mes_pesquisa,preco_mais_caro,posto_mais_caro,preco_mais_barato,posto_mais_barato,amplitude_rs,postos_pesquisados,primeira_coleta,ultima_coleta
0,2026-06,4.790,ORGANIZACOES NOVO BELVEDERE LTDA,3.500,POSTO ALLGAS LTDA,1.290,49,2026-06-01,2026-06-30


## Fechando
**Próximos passos:** estas consultas viram o gabarito do `eval/golden_set.csv`. Elas seguem as regras R01–R14 do
`semantic_layer.yaml` — se uma regra mudar lá, o SQL daqui precisa mudar junto (e o notebook ser executado de novo).

In [12]:
con.close()